# 02 - Data engineering and preprocessing
**Milestone 2 (spec 3.2)** | Owner: Data Engineer (role 2). Run `01_eda.ipynb` first (this notebook rebuilds the inventory itself if `inventory.csv` is missing).

In [ ]:
import os, sys
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/aie231-project/notebooks
sys.path.insert(0, os.path.abspath('..'))
!pip -q install kagglehub
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import *

In [ ]:
inv_path = 'inventory.csv'
ROOT = find_data_root(download())     # fresh Colab session => fresh path
if os.path.exists(inv_path):
    inv = rebase_paths(pd.read_csv(inv_path), ROOT)   # saved paths are stale in a new session
else:
    inv = build_inventory(ROOT); inv.to_csv(inv_path, index=False)
print('Raw images:', len(inv))

## 1. Cleaning: exact duplicates
Two files with identical bytes are the same image. If one copy is in train and one in test the model is graded on something it has seen (**leakage**). We keep one copy. If identical bytes had *different labels* we would drop them all (ambiguous label). Missing values do not exist here (every image has a folder label), so there is nothing to impute: that is itself a finding to state in the report.

In [ ]:
clean = drop_exact_duplicates(inv)
print('Removed', len(inv) - len(clean), 'duplicate/conflicting images ->', len(clean), 'left')

## 2. Split design: stratified, grouped 70/15/15
Why not the official folders? The official `val/` has 16 images (a validation score from 16 images is noise), and the official train/test ratios differ. So we pool everything and re-split.

* **Stratified:** every split keeps the same Normal:Pneumonia ratio.
* **Grouped:** all images of one pneumonia patient (same `personNNN` in the file name) go to the same split. Otherwise the model sees the same child in train and test.
* **Limitation to state in the report:** NORMAL file names carry no patient id, so Normal images are treated as independent. A Normal patient could appear in two splits.
* **Method:** `StratifiedGroupKFold` with 20 folds (seeded), 14 folds -> train, 3 -> val, 3 -> test = 70/15/15. The **test set is touched once, at the very end.** All model selection uses val only.

In [ ]:
df = make_splits(clean, seed=SEED)
check_no_leakage(df)          # raises if any patient / file / hash is in two splits
print('Leakage check passed.')
summary = df.groupby(['split', 'label_name']).size().unstack()
summary['total'] = summary.sum(axis=1); summary['pneumonia_ratio'] = (summary['PNEUMONIA'] / summary['NORMAL']).round(2)
summary['share_%'] = (100 * summary['total'] / len(df)).round(1)
display(summary.loc[['train', 'val', 'test']])
df[['path', 'label', 'label_name', 'orig_split', 'split', 'group', 'md5']].to_csv('splits.csv', index=False)

**Check by eye:** the `pneumonia_ratio` should be nearly equal in the three rows, and shares close to 70/15/15.

## 3. Imbalance: measure, then handle
Technique 1 = **class weights** computed from the TRAIN set only: w_c = N / (2 * n_c). The loss for a Normal image is multiplied by the larger weight, so a missed Normal costs more during training and the model cannot just predict "pneumonia".
Technique 2 = **augmentation** (below) applied to the training set only.
The effect on per-class recall is measured in the ablations (Milestone 6): with vs without class weights, with vs without augmentation.

In [ ]:
train_df = df[df.split == 'train']
cw = class_weights(train_df)
print('Class weights (0=NORMAL, 1=PNEUMONIA):', {k: round(v, 3) for k, v in cw.items()})

## 4. Pixel scaling and resize
Every image -> 224x224 (VGG16/ResNet-50 input size), converted to 3 channels (grey X-ray replicated), kept as float [0,255] by the data pipeline. **Scaling happens inside the model** (`Rescaling(1/255)` for the custom CNN, the network's own `preprocess_input` for VGG16/ResNet-50). Reason: no scaler to "fit", nothing to leak, and `predict.py` reuses the exact same model so there is no duplicated preprocessing between training and inference (a rubric item).

## 5. Augmentation (train only)
Rotation (+-4%, about 14 degrees), zoom (10%), contrast (15%). **No horizontal flip**: a flipped chest X-ray moves the heart to the other side, which is anatomically unrealistic. Validation and test are never augmented.

In [ ]:
import tensorflow as tf
ds_plain = make_dataset(train_df.head(8), batch_size=8, augment=False)
ds_aug   = make_dataset(train_df.head(8), batch_size=8, augment=True)
(xp, _), (xa, _) = next(iter(ds_plain)), next(iter(ds_aug))
fig, ax = plt.subplots(2, 6, figsize=(15, 5.5))
for i in range(6):
    ax[0, i].imshow(xp[i].numpy().astype('uint8')); ax[0, i].axis('off'); ax[0, i].set_title('original')
    ax[1, i].imshow(np.clip(xa[i].numpy(), 0, 255).astype('uint8')); ax[1, i].axis('off'); ax[1, i].set_title('augmented')
plt.tight_layout(); plt.show()

## 6. Build the three datasets and sanity-check shapes
Shuffle only the training set. `drop`-free batching with prefetch keeps the Colab GPU busy.

In [ ]:
BATCH = 32
train_ds = make_dataset(train_df, BATCH, training=True, augment=True)
val_ds   = make_dataset(df[df.split == 'val'],  BATCH)
test_ds  = make_dataset(df[df.split == 'test'], BATCH)    # not used until the final evaluation
x, y = next(iter(train_ds))
print('batch images', x.shape, x.dtype, '| range', float(tf.reduce_min(x)), '-', float(tf.reduce_max(x)), '| labels', y.shape, 'mean', float(tf.reduce_mean(y)))

## 7. Findings -> decisions (for the report, section 4.1)
| Decision | Reason |
|---|---|
| Own stratified 70/15/15 split with fixed seed 42 | official val has 16 images; keep class ratio in each split |
| Group by patient id, drop exact duplicates | prevents the same child / file in train and test |
| Class weights + augmentation, train only | handle imbalance; show per-class recall effect in ablations |
| Resize 224, scale inside the model | transfer-model input size; no leakage; one shared pipeline |
| No flips | anatomical realism |
| Test set untouched until the end | spec rule; any use for tuning costs marks |